In [1]:
pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 9.9 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


### 1. Linear Programming
- identify decision variables
- set objective function
- add constraints

#### Trail Mix

In [14]:
from gurobipy import *

## DAHLBY OUTFITTERS EXAMPLE##

# Create a new model
m = Model("Example 2")

In [15]:

# Create variables
s = m.addVar(vtype=GRB.CONTINUOUS, lb=0,name="Seeds")
r = m.addVar(vtype=GRB.CONTINUOUS, lb=0,name="Raisins")
f = m.addVar(vtype=GRB.CONTINUOUS, lb=0,name="Flakes")
p = m.addVar(vtype=GRB.CONTINUOUS, lb=0,name="Pecans")
w = m.addVar(vtype=GRB.CONTINUOUS, lb=0,name="Walnuts")


In [16]:

# Set objective
m.setObjective(4*s + 5*r + 3*f + 7*p + 6*w, GRB.MINIMIZE)


In [17]:
# Add constraints
m.addConstr(10*s + 20*r + 10*f + 30*p + 20*w >= 20, "Vitamins")
m.addConstr(5*s + 7*r + 4*f + 9*p + 2*w >= 10, "Minerals")
m.addConstr(1*s + 4*r + 10*f + 2*p + 1*w >= 15, "Protein")
m.addConstr(500*s + 450*r + 160*f + 300*p + 500*w >= 600, "Calories")

# Optimize
m.optimize()

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 24.2.0 24C101)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4 rows, 5 columns and 20 nonzeros
Model fingerprint: 0xbd9b0c7e
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [3e+00, 7e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 6e+02]
Presolve time: 0.00s
Presolved: 4 rows, 5 columns, 20 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   9.250000e+01   0.000000e+00      0s
       3    7.5357995e+00   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.00 seconds (0.00 work units)
Optimal objective  7.535799523e+00


In [18]:
for v in m.getVars():
    print(f"{v.varname,v.x}")

('Seeds', 0.47732696897374705)
('Raisins', 0.33412887828162285)
('Flakes', 1.3186157517899761)
('Pecans', 0.0)
('Walnuts', 0.0)


In [19]:
# Print results for each constraint
for v in m.getVars():
    print('%s: %g' % (v.varName, v.x))

Seeds: 0.477327
Raisins: 0.334129
Flakes: 1.31862
Pecans: 0
Walnuts: 0


In [20]:
print(m.getVars())

[<gurobi.Var Seeds (value 0.47732696897374705)>, <gurobi.Var Raisins (value 0.33412887828162285)>, <gurobi.Var Flakes (value 1.3186157517899761)>, <gurobi.Var Pecans (value 0.0)>, <gurobi.Var Walnuts (value 0.0)>]


#### Shadow Price

In [21]:
shadow_prices = m.getAttr('Pi')
m.printAttr('Pi')


  Constraint           Pi 
-------------------------
    Minerals     0.490453 
     Protein    0.0560859 
    Calories   0.00298329 


In [22]:
slack_valueus=m.getAttr('slack')
m.printAttr('slack')


  Constraint        slack 
-------------------------
    Vitamins       -4.642 


### Blended Models
- do the algebra first, and then use it as a constraint

In [24]:
m = Model("Blending Model")


In [25]:
# Create variables
b = m.addVar(vtype=GRB.CONTINUOUS, lb=0,name="Brazilian")
c = m.addVar(vtype=GRB.CONTINUOUS, lb=0,name="Colombian")
p = m.addVar(vtype=GRB.CONTINUOUS, lb=0,name="Peruvian")


In [26]:
# Set objective - this is the cost
m.setObjective(0.5*b + 0.6*c + 0.7*p, GRB.MINIMIZE)


In [27]:
# Add constraints - the algebra, and the avaialbe coffee
m.addConstr(-3*b - 18*c + 7*p >= 0, name='Aroma')
m.addConstr(-1*b + 4*c + 2*p >= 0, name='Strength')

# available coffee
m.addConstr(b <= 1500000, name='Brazilian Avail')
m.addConstr(c <= 1200000, name='Colombian Avail')
m.addConstr(p <= 2000000, name='Peruvian Avail')

# max production
m.addConstr(b + c + p == 4000000, name='Blend Total')

<gurobi.Constr *Awaiting Model Update*>

In [28]:
# Optimize
m.optimize()

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 24.2.0 24C101)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 6 rows, 3 columns and 12 nonzeros
Model fingerprint: 0x1b0e3b58
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [5e-01, 7e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+06, 4e+06]
Presolve removed 6 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.4480000e+06   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.448000000e+06


In [29]:
# Print results for each constraint
for v in m.getVars():
    print('%s: %g' % (v.varName, v.x))

Brazilian: 1.5e+06
Colombian: 520000
Peruvian: 1.98e+06


In [30]:
# Print result for Objective using optimized constraints
print('Obj: %g' % m.objVal)

Obj: 2.448e+06


In [31]:
# Get shadow prices
m.printAttr('Pi')


  Constraint           Pi 
-------------------------
       Aroma        0.004 
Brazilian Avail        -0.16 
 Blend Total        0.672 


In [32]:
# Ger reduce cost
m.printAttr('Rc')


    Variable           Rc 
-------------------------
